In [1]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
import wandb
from wandb.integration.sb3 import WandbCallback
import os
import glob
import psutil

import subprocess
import imageio_ffmpeg

In [2]:

# List all available environments
print(sorted(gym.registry.keys()))

['Acrobot-v1', 'Ant-v2', 'Ant-v3', 'Ant-v4', 'Ant-v5', 'BipedalWalker-v3', 'BipedalWalkerHardcore-v3', 'Blackjack-v1', 'CarRacing-v3', 'CartPole-v0', 'CartPole-v1', 'CliffWalking-v1', 'CliffWalkingSlippery-v1', 'FrozenLake-v1', 'FrozenLake8x8-v1', 'GymV21Environment-v0', 'GymV26Environment-v0', 'HalfCheetah-v2', 'HalfCheetah-v3', 'HalfCheetah-v4', 'HalfCheetah-v5', 'Hopper-v2', 'Hopper-v3', 'Hopper-v4', 'Hopper-v5', 'Humanoid-v2', 'Humanoid-v3', 'Humanoid-v4', 'Humanoid-v5', 'HumanoidStandup-v2', 'HumanoidStandup-v4', 'HumanoidStandup-v5', 'InvertedDoublePendulum-v2', 'InvertedDoublePendulum-v4', 'InvertedDoublePendulum-v5', 'InvertedPendulum-v2', 'InvertedPendulum-v4', 'InvertedPendulum-v5', 'LunarLander-v3', 'LunarLanderContinuous-v3', 'MountainCar-v0', 'MountainCarContinuous-v0', 'Pendulum-v1', 'Pusher-v2', 'Pusher-v4', 'Pusher-v5', 'Reacher-v2', 'Reacher-v4', 'Reacher-v5', 'Swimmer-v2', 'Swimmer-v3', 'Swimmer-v4', 'Swimmer-v5', 'Taxi-v4', 'Walker2d-v2', 'Walker2d-v3', 'Walker2d-v4'

In [3]:
class StreamingVideoCallback(BaseCallback):
    def __init__(self, video_folder, record_every=100_000, video_length=200, fps=30):
        super().__init__()
        self.video_folder = video_folder
        self.record_every = record_every
        self.video_length = video_length
        self.fps = fps
        self.next_video_at = 1
        self.encoder = None
        self.frames_written = 0
        self.current_video_path = None
        self.completed_videos = []

    def _start_video(self, frame):
        os.makedirs(self.video_folder, exist_ok=True)
        height, width = frame.shape[:2]
        self.current_video_path = os.path.join(
            self.video_folder,
            f"rl-video-step-{self.num_timesteps}.mp4",
        )
        self.encoder = subprocess.Popen(
            [
                imageio_ffmpeg.get_ffmpeg_exe(),
                "-y", "-loglevel", "error",
                "-f", "rawvideo",
                "-pix_fmt", "rgb24",
                "-s:v", f"{width}x{height}",
                "-r", str(self.fps),
                "-i", "-",
                "-an",
                "-c:v", "libx264",
                "-preset", "ultrafast",
                "-pix_fmt", "yuv420p",
                "-movflags", "+faststart",
                self.current_video_path,
            ],
            stdin=subprocess.PIPE,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        self.frames_written = 0

    def _finish_video(self):
        if self.encoder is not None:
            self.encoder.stdin.close()
            return_code = self.encoder.wait()
            self.encoder = None
            if return_code:
                raise RuntimeError(f"FFmpeg exited with code {return_code}")
            self.completed_videos.append(self.current_video_path)
            self.current_video_path = None

    def _on_step(self):
        if self.encoder is None:
            if self.num_timesteps < self.next_video_at:
                return True
            self._start_video(self.training_env.render())
            self.next_video_at = self.num_timesteps + self.record_every

        frame = self.training_env.render()
        self.encoder.stdin.write(frame.tobytes())
        self.frames_written += 1

        if self.frames_written >= self.video_length:
            self._finish_video()

        return True

    def _on_training_end(self):
        self._finish_video()


class VideoLoggerCallback(BaseCallback):
    def __init__(self, video_recorder, verbose=0):
        super().__init__(verbose)
        self.video_recorder = video_recorder
        self.logged_videos = set()

    def _log_completed_videos(self):
        for video_path in self.video_recorder.completed_videos:
            if video_path not in self.logged_videos and os.path.getsize(video_path) > 0:
                wandb.log({"training_video": wandb.Video(video_path, format="mp4")})
                self.logged_videos.add(video_path)

    def _on_step(self) -> bool:
        self._log_completed_videos()
        return True

    def _on_training_end(self):
        self._log_completed_videos()

In [ ]:
# Initialize a new wandb run


config = {
    "total_timesteps": 30_000_000,
    "env_name": "Humanoid-v5",
    "paralel_envs": 8,
    "algorithm": "PPO",
    "policy_type": "MlpPolicy",
    "learning_rate": 0.0003,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "n_steps": 2048,
    "n_epochs": 10,
}



run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="RL_locomotion_FEKTBOT",
    # Set the wandb project where this run will be logged.
    project="RL_locomotion_FEKTBOT_project",
    # Track hyperparameters and run metadata.
    config=config,
    sync_tensorboard=True,   
    monitor_gym=True,        
    save_code=True,
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/aizej/.netrc.
wandb: Currently logged in as: aizej (RL_locomotion_FEKTBOT) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Artifact "source-RL_locomotion_FEKTBOT_project-_home_aizej_Desktop_RL_lokomoce_FEKTBOT_testing_gymnasium.ipynb" already exists with the same content. No new version will be created.


In [5]:
video_recorder = StreamingVideoCallback(video_folder=f"videos/{run.id}")
callback = CallbackList([
    WandbCallback(
        gradient_save_freq=0,
        model_save_path=f"models/{run.id}",
        verbose=2,
    ),
    video_recorder,
    VideoLoggerCallback(video_recorder),
])

In [ ]:
def make_single_env():
    env = gym.make(config["env_name"], render_mode="rgb_array")
    env = gym.wrappers.RecordEpisodeStatistics(env)
    return env


# Create a vectorized environment
env = SubprocVecEnv([make_single_env for _ in range(config["paralel_envs"])])

In [7]:



model = PPO("MlpPolicy", env,
            learning_rate=config["learning_rate"],
            gamma=config["gamma"],
            gae_lambda=config["gae_lambda"],
            n_steps=config["n_steps"],
            n_epochs=config["n_epochs"],
            verbose=1, tensorboard_log=f"runs/{run.id}")



model.learn(total_timesteps=config["total_timesteps"], 
            callback=callback)



Using cpu device
Logging to runs/dr94l060/PPO_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21.6     |
|    ep_rew_mean     | 96.7     |
| time/              |          |
|    fps             | 1483     |
|    iterations      | 1        |
|    time_elapsed    | 11       |
|    total_timesteps | 16384    |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 22         |
|    ep_rew_mean          | 98.4       |
| time/                   |            |
|    fps                  | 1885       |
|    iterations           | 2          |
|    time_elapsed         | 17         |
|    total_timesteps      | 32768      |
| train/                  |            |
|    approx_kl            | 0.02040162 |
|    clip_fraction        | 0.246      |
|    clip_range           | 0.2        |
|    entropy_loss         | -24.1      |
|    explained_variance   | -0.00383  

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7fd737876900>> (for post_run_cell), with arguments args (<ExecutionResult object at 7fd7377b9e50, execution_count=7 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7fd7377b9f40, raw_cell="


model = PPO("MlpPolicy", env,
            learn.." transformed_cell="model = PPO("MlpPolicy", env,
            learning.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/home/aizej/Desktop/RL_lokomoce_FEKTBOT/testing_gymnasium.ipynb#W6sZmlsZQ%3D%3D cell_meta={'cellId': 'vscode-notebook-cell:/home/aizej/Desktop/RL_lokomoce_FEKTBOT/testing_gymnasium.ipynb#W6sZmlsZQ%3D%3D'}> result=None>,),kwargs {}:


MailboxClosedError: 